# conv-windowing-1d composite — cx12: conv1d from scratch — as_strided windowing + einsum

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `conv-windowing-1d`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-windowing-1d"
DD_ATOM_IDS = ["as-strided-windowing", "conv-windowing-1d"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "CNN: 1-D conv windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

This is the ARENA `conv1d_minimal` recipe in one function. It decomposes a 1-D conv into TWO ops:
1. **Windowing** (atom: `as-strided-windowing`) — build a `(B, IC, OW, K)` view onto `x` with strides `(s_b, s_ic, s_w, s_w)`. No copy.
2. **Einsum contraction** (atom: `conv-windowing-1d` — the windowed-view-then-einsum pattern) — `einops.einsum(win, weight, 'b ic ow kw, oc ic kw -> b oc ow')`. This contracts the `IC` and `KW` axes simultaneously, dotting each window against each filter.

**Why this is exactly `F.conv1d`.** Each output cell `y[b, oc, ow]` is the sum over `ic, kw` of `x[b, ic, ow + kw] * weight[oc, ic, kw]`. The windowing arranges `x` so position `(b, ic, ow, kw)` IS `x[b, ic, ow + kw]`. The einsum then evaluates the conv formula in one call.

**Anatomy.**
- `OW = W - KW + 1` (stride 1, no pad).
- `s_b, s_ic, s_w = x.stride()`.
- `win = x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, s_w, s_w))`.
- `y = einops.einsum(win, weight, 'b ic ow kw, oc ic kw -> b oc ow')`.

**Why care.** This is the from-scratch building block ARENA uses to *implement* conv before touching `F.conv1d`. Padding and stride extensions slot in on top: pad x first (cx8); multiply the OW stride by S (cx9). This drill is the unpadded, stride-1 base case.

### Composite Exercise — conv1d from scratch — as_strided windowing + einsum

**Atoms exercised together**: `as-strided-windowing`, `conv-windowing-1d`

Implement `cx12_conv1d_from_scratch(x, weight)` — a from-scratch 1-D convolution that uses ONLY `as_strided` and `einops.einsum`. Do NOT call `F.conv1d`. Stride 1, no padding.

- `x`: float tensor of shape `(B, IC, W)`.
- `weight`: float tensor of shape `(OC, IC, KW)`.
- Return: float tensor of shape `(B, OC, OW)` where `OW = W - KW + 1`.

Two-step recipe:
1. **Window** `x` into a `(B, IC, OW, KW)` view via `as_strided`. Read `x.stride()` for the source strides; the OW and KW axes both get stride `s_w` (stride-1 windowing pattern).
2. **Einsum** the view against `weight`: `'b ic ow kw, oc ic kw -> b oc ow'`. This contracts the `IC` and `KW` axes in one shot.

The test:
- Cross-checks against `F.conv1d(x, weight)` to fp tolerance.
- Verifies the intermediate windowed view shares storage with `x` (no copy in step 1).
- Probes edge cases: `KW == W` (single window), `KW == 1` (just channel mix), batch + multi-channel.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx12_conv1d_from_scratch(x, weight):
    raise NotImplementedError

def cx12_window_only(x, KW):
    """Helper exposing the windowed view so the test can verify the no-copy property."""
    raise NotImplementedError

def _test_cx12():
    from torch.nn import functional as F

    # Case A: hand-built — single batch, single channel, kw=3.
    x = t.arange(1.0, 6.0).reshape(1, 1, 5).contiguous()  # [1,2,3,4,5]
    weight = t.tensor([[[1.0, 0.0, -1.0]]])  # (1, 1, 3)
    y = cx12_conv1d_from_scratch(x, weight)
    assert tuple(y.shape) == (1, 1, 3), f'shape: {tuple(y.shape)}'
    # Each output = x[k] - x[k+2]. y = [1-3, 2-4, 3-5] = [-2, -2, -2].
    assert t.allclose(y, t.tensor([[[-2.0, -2.0, -2.0]]])), f'y={y}'

    # Case B: no-copy windowing — the intermediate view must alias x.
    x = t.arange(1.0, 11.0).reshape(1, 1, 10).contiguous()
    win = cx12_window_only(x, KW=3)
    assert tuple(win.shape) == (1, 1, 8, 3)
    assert win.data_ptr() == x.data_ptr(), 'windowed view must share storage with x'
    for k in range(8):
        assert t.allclose(win[0, 0, k], x[0, 0, k:k+3])

    # Case C: multi-channel, multi-filter cross-check against F.conv1d.
    rng = t.Generator().manual_seed(12)
    B, IC, W, OC, KW = 2, 3, 16, 4, 5
    x = t.randn(B, IC, W, generator=rng)
    weight = t.randn(OC, IC, KW, generator=rng)
    y_manual = cx12_conv1d_from_scratch(x, weight)
    y_native = F.conv1d(x, weight)
    assert tuple(y_manual.shape) == (B, OC, W - KW + 1)
    assert t.allclose(y_manual, y_native, atol=1e-4), (
        f'from-scratch conv1d disagrees with F.conv1d (max diff = {(y_manual - y_native).abs().max()})'
    )

    # Case D: KW == W — single output cell, all of x dotted against the kernel.
    B, IC, W, OC = 2, 3, 5, 4
    x = t.randn(B, IC, W, generator=rng)
    weight = t.randn(OC, IC, W, generator=rng)
    y = cx12_conv1d_from_scratch(x, weight)
    assert tuple(y.shape) == (B, OC, 1)
    y_native = F.conv1d(x, weight)
    assert t.allclose(y, y_native, atol=1e-4)

    # Case E: KW == 1 — pure pointwise channel mix, identical to a 1x1 conv.
    B, IC, W, OC = 1, 5, 7, 3
    x = t.randn(B, IC, W, generator=rng)
    weight = t.randn(OC, IC, 1, generator=rng)
    y = cx12_conv1d_from_scratch(x, weight)
    assert tuple(y.shape) == (B, OC, W)
    y_native = F.conv1d(x, weight)
    assert t.allclose(y, y_native, atol=1e-4)

    # Case F: fuzz over many random configs.
    for B, IC, W, OC, KW in [
        (1, 1, 8, 1, 3),
        (3, 2, 20, 5, 4),
        (1, 8, 12, 1, 1),
        (2, 4, 30, 8, 7),
    ]:
        x = t.randn(B, IC, W, generator=rng)
        weight = t.randn(OC, IC, KW, generator=rng)
        y_manual = cx12_conv1d_from_scratch(x, weight)
        y_native = F.conv1d(x, weight)
        assert t.allclose(y_manual, y_native, atol=1e-4), (
            f'mismatch for B={B} IC={IC} W={W} OC={OC} KW={KW}'
        )
    _dd_passed.add('cx12')

_test_cx12()

<details><summary>Show solution — cx12</summary>

```python
def cx12_window_only(x, KW):
    B, IC, W = x.shape
    OW = W - KW + 1
    s_b, s_ic, s_w = x.stride()
    # Atom A (as-strided-windowing): stride-1 pattern, trailing (s_w, s_w).
    return x.as_strided(
        size=(B, IC, OW, KW),
        stride=(s_b, s_ic, s_w, s_w),
    )

def cx12_conv1d_from_scratch(x, weight):
    OC, IC, KW = weight.shape
    win = cx12_window_only(x, KW)
    # Atom B (conv-windowing-1d): contract IC and KW in one einsum.
    return einops.einsum(win, weight, 'b ic ow kw, oc ic kw -> b oc ow')
```

Two atoms, two lines. The windowing is the load-bearing trick — once `x` is reshaped so position `(b, ic, ow, kw)` holds `x[b, ic, ow + kw]`, the conv reduces to a single einsum that contracts both IC and KW. The view shares storage with x — no allocation between input and einsum. Extending this to stride > 1 means multiplying the OW-axis stride by S (cx9); extending to non-zero padding means padding x first (cx8). This drill is the unpadded, stride-1 base case from which ARENA's full conv1d is built.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["PyTorch: as_strided windowing", "CNN: 1-D conv windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()